<a href="https://colab.research.google.com/github/sarkarsushmita408-ctrl/quPSI/blob/main/quPSI_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matplotlib-venn

In [ ]:
!pip install qiskit qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.8 MB/s eta 0:00:00


In [ ]:
import qiskit
import qiskit_aer

print("Qiskit Version:", qiskit.__version__)
print("Qiskit_Aer:", qiskit_aer.__version__)

Qiskit Version: 2.5.2
Qiskit_Aer: 0.17.2


In [ ]:
from __future__ import annotations

import hashlib
import math
import random
from dataclasses import dataclass, field
from typing import Iterable, Optional

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# **quPSI with Time and Space Complexity**

In [ ]:
"""
quPSI: A QKD-Enabled Quantum Secure Mutual Private Set Intersection Protocol
============================================================================
Single-file implementation -- quantum layer (BB84 via Qiskit) + classical
layer (Bloom filters, Trusted Authority, protocol driver) -- of the
construction in Section 3 of:

  "A QKD-Enabled Quantum Secure mPSI Framework for Private Law Enforcement
  Data Intersection" (Kumar, Sarkar, Dey, Srivastava, Debnath, Kouichi).

The file is organized top-to-bottom as two layers that are cleanly
separated but wired together:

  PART A -- QUANTUM LAYER   (Sec. 2.2): BB84 QKD, simulated with real
            Qiskit circuits (state preparation, transmission, measurement,
            sifting, public QBER estimation, abort-on-high-error).

  PART B -- CLASSICAL LAYER (Sec. 2.1, 2.3, 3): Boolean AND, Bloom
            filters, the Trusted Authority, John/David as parties, and
            the three quPSI phases -- quPSI.kg, quPSI.int, quPSI.comp --
            which consume keys produced by Part A.

  PART C -- PERFORMANCE INSTRUMENTATION: a Metrics collector threaded
            through both layers that records, for every phase
            (quPSI.kg / quPSI.int / quPSI.comp):
              - computation time, split into quantum (BB84 circuit
                simulation) and classical (sifting, hashing, Bloom
                filter ops, XOR, AND, membership tests);
              - communication size, i.e. how many qubits/bits cross
                each link (Alice<->Bob quantum channel, Alice<->Bob and
                Alice/Bob<->TA classical channels), with an optional
                bandwidth model to turn size into an estimated
                communication time.
            run_quPSI(..., collect_metrics=True) returns this alongside
            the intersection result, and print_performance_report()
            renders it as a table.

Design notes / simplifying assumptions:

  * QKD (Sec. 2.2) is instantiated with an actual BB84 simulation on
    Qiskit: Alice prepares qubits in randomly chosen bases encoding
    random bits, Bob measures in independently random bases, and the
    two sides sift, publicly sample a subset to estimate the QBER, and
    abort if it exceeds the standard 11% threshold (Sec. 2.2: "if the
    error exceeds a predefined threshold, the protocol aborts"). Each
    of the three key-sharing steps in quPSI.kg -- (John,TA), (David,TA),
    (John,David) -- runs its own independent BB84 session. Real photon
    transport over an optical channel obviously can't happen inside
    this process, so the "channel" is Qiskit's Aer simulator (method=
    matrix_product_state, which is efficient here because a BB84 round
    has no entangling gates); everything downstream of key establishment
    (masking, Bloom filters, XOR-masking, TA's f_AND, local recovery) is
    exactly as specified in the paper.
  * The hash function H and hash family {h_0,...,h_{k-1}} are modeled as
    a random oracle using SHA-256, exactly as assumed in Sec. 4.2.
  * The Trusted Authority (TA) only ever sees XOR-masked Bloom filters,
    mirroring the semi-trusted TA of the protocol (it cannot recover
    BF_X, BF_Y, or X, Y from what it receives).

Run this file directly to see three demos: the paper's own toy example
(Sec. 3.2), a larger random-set example, and a noisy-channel example
where BB84 correctly aborts before any Bloom filters are exchanged.
"""

from __future__ import annotations

import hashlib
import math
import random
import time
from collections import defaultdict
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Iterable, Optional

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error


# =============================================================================
# PART C -- PERFORMANCE INSTRUMENTATION
# (defined first because Parts A and B both record into it)
# =============================================================================
#
# Two things are tracked, per protocol phase ("kg", "int", "comp") and,
# within "kg", per QKD link ("John-TA", "David-TA", "John-David"):
#
#   1. COMPUTATION TIME -- actual wall-clock seconds spent, split into
#      "quantum" (building + simulating BB84 circuits on Aer -- this is
#      the stand-in for time a real quantum device/simulator would take
#      to prepare, transmit and measure photons) and "classical"
#      (sifting, QBER estimation, hashing, Bloom filter insert/query,
#      XOR masking, f_AND, membership tests).
#
#   2. COMMUNICATION SIZE -- bits/qubits that would cross the wire
#      between real, physically-separated parties, recorded per logical
#      link. This is a genuine count of protocol messages (qubits sent,
#      classical bits exchanged for basis reconciliation and QBER
#      sampling, masked Bloom filters sent to/from the TA) -- it is
#      *not* inferred from timing. An optional bandwidth model
#      (bits/sec for the classical channel, photons/sec i.e. Hz for the
#      quantum channel) turns that size into an estimated transmission
#      time, separate from and additional to computation time.
# =============================================================================

DEFAULT_CLASSICAL_BANDWIDTH_BPS = 1_000_000_000     # 1 Gbps classical link
DEFAULT_QUANTUM_CHANNEL_RATE_HZ = 10_000_000        # 10 MHz photon pulse rate


@contextmanager
def _null_context():
    """No-op context manager used when metrics collection is disabled,
    so instrumented code doesn't need an `if metrics:` branch at every
    call site."""
    yield


@dataclass
class LinkComm:
    """Communication tally for one logical link (e.g. 'John-TA')."""
    quantum_qubits: int = 0     # qubits carried on the quantum channel
    classical_bits: int = 0     # bits carried on the (public) classical channel


class Metrics:
    """Collects computation time and communication size for one run of
    run_quPSI, broken down by phase and by link, plus helpers to turn
    communication size into an estimated communication time.
    """

    PHASES = ("kg", "int", "comp")

    def __init__(self,
                 classical_bandwidth_bps: float = DEFAULT_CLASSICAL_BANDWIDTH_BPS,
                 quantum_channel_rate_hz: float = DEFAULT_QUANTUM_CHANNEL_RATE_HZ):
        self.classical_bandwidth_bps = classical_bandwidth_bps
        self.quantum_channel_rate_hz = quantum_channel_rate_hz

        # computation_time[phase]["quantum" | "classical"] -> seconds
        self.computation_time = {p: defaultdict(float) for p in self.PHASES}
        # communication[phase][link_name] -> LinkComm
        self.communication = {p: defaultdict(LinkComm) for p in self.PHASES}

    # -- recording -----------------------------------------------------
    @contextmanager
    def time_block(self, phase: str, kind: str):
        """Context manager: `with metrics.time_block("kg", "quantum"): ...`
        adds the wall-clock duration of the block to computation_time.
        """
        t0 = time.perf_counter()
        try:
            yield
        finally:
            self.computation_time[phase][kind] += time.perf_counter() - t0

    def record_comm(self, phase: str, link: str,
                     quantum_qubits: int = 0, classical_bits: int = 0) -> None:
        entry = self.communication[phase][link]
        entry.quantum_qubits += quantum_qubits
        entry.classical_bits += classical_bits

    # -- aggregation -----------------------------------------------------
    def phase_computation_seconds(self, phase: str) -> float:
        return sum(self.computation_time[phase].values())

    def total_computation_seconds(self) -> float:
        return sum(self.phase_computation_seconds(p) for p in self.PHASES)

    def phase_communication_bits(self, phase: str) -> tuple:
        """Returns (quantum_qubits, classical_bits) summed over all
        links in that phase."""
        q = sum(l.quantum_qubits for l in self.communication[phase].values())
        c = sum(l.classical_bits for l in self.communication[phase].values())
        return q, c

    def estimated_communication_seconds(self, quantum_qubits: int, classical_bits: int) -> float:
        """Estimated wire time for the given amount of quantum + classical
        traffic, under this Metrics object's bandwidth model. The two
        channels are physically distinct, so they run in parallel; the
        estimate is the slower (bottleneck) of the two, as is standard
        for reporting an end-to-end transmission time.
        """
        t_quantum = quantum_qubits / self.quantum_channel_rate_hz if quantum_qubits else 0.0
        t_classical = classical_bits / self.classical_bandwidth_bps if classical_bits else 0.0
        return max(t_quantum, t_classical)

    def phase_communication_seconds(self, phase: str) -> float:
        q, c = self.phase_communication_bits(phase)
        return self.estimated_communication_seconds(q, c)

    def total_communication_bits(self) -> tuple:
        q = sum(self.phase_communication_bits(p)[0] for p in self.PHASES)
        c = sum(self.phase_communication_bits(p)[1] for p in self.PHASES)
        return q, c

    def total_communication_seconds(self) -> float:
        q, c = self.total_communication_bits()
        return self.estimated_communication_seconds(q, c)


def print_performance_report(metrics: "Metrics") -> None:
    """Pretty-prints a per-phase breakdown of computation time,
    communication size (qubits and bytes, in their own columns), and
    estimated communication time, plus grand totals.
    """
    phase_labels = {"kg": "quPSI.kg (BB84 key generation)",
                     "int": "quPSI.int (masked BF exchange via TA)",
                     "comp": "quPSI.comp (local intersection recovery)"}

    col = "{:<40s}{:>12s}{:>12s}{:>12s}{:>14s}"
    rule = "-" * 90
    print(rule)
    print(col.format("Phase", "Comp (s)", "Qubits", "Bytes", "Comm (s)"))
    print(rule)
    for phase in Metrics.PHASES:
        comp_t = metrics.phase_computation_seconds(phase)
        q_bits, c_bits = metrics.phase_communication_bits(phase)
        comm_t = metrics.phase_communication_seconds(phase)
        print(col.format(phase_labels[phase], f"{comp_t:.4f}",
                          str(q_bits) if q_bits else "-",
                          f"{c_bits / 8:.0f}" if c_bits else "-",
                          f"{comm_t:.6f}"))

    print(rule)
    total_comp = metrics.total_computation_seconds()
    total_q, total_c = metrics.total_communication_bits()
    total_comm_t = metrics.total_communication_seconds()
    print(col.format("TOTAL", f"{total_comp:.4f}",
                      str(total_q), f"{total_c / 8:.0f}", f"{total_comm_t:.6f}"))
    print(rule)

    # Breakdown of computation time by quantum vs classical.
    q_time = sum(metrics.computation_time[p].get("quantum", 0.0) for p in Metrics.PHASES)
    c_time = sum(metrics.computation_time[p].get("classical", 0.0) for p in Metrics.PHASES)
    print(f"  computation time -- quantum (BB84 sim): {q_time:.4f} s, "
          f"classical: {c_time:.4f} s")
    print(f"  bandwidth model  -- classical: {metrics.classical_bandwidth_bps/1e9:.2f} Gbps, "
          f"quantum channel: {metrics.quantum_channel_rate_hz/1e6:.1f} MHz pulse rate")


# =============================================================================
# PART A -- QUANTUM LAYER: BB84 Quantum Key Distribution (Sec. 2.2)
# =============================================================================
#
# Implements the BB84 protocol referenced in Sec. 2.2 ("A widely studied
# class of QKD protocols is the prepare-and-measure paradigm, exemplified
# by the BB84 protocol") with real Qiskit circuits and the paper's four
# textbook sub-steps:
#
#   1. Quantum transmission  (state prep + measurement, simulated by Qiskit)
#   2. Sifting                (discard qubits where bases disagree)
#   3. Parameter estimation   (public sample -> estimate QBER, abort if too high)
#   4. Key extraction         (remaining sifted bits become the shared key)
#
# Circuit encoding (per photon/qubit):
#   Alice: bit=1 -> X gate; basis=1 (X-basis) -> H gate afterwards.
#          (basis=0 is the Z/computational basis: |0>, |1>)
#          (basis=1 is the X/Hadamard basis:      |+>, |->)
#   Bob:   basis=1 -> apply H before measuring in the computational basis
#          (the standard trick for measuring in the X basis).
#
# All qubits in a round are independent (no entangling gates), so a
# single circuit with many qubits is simulated efficiently with Aer's
# matrix-product-state backend (linear cost, not the 2^n cost of a naive
# statevector simulation).
# =============================================================================


@dataclass
class BB84Stats:
    """Diagnostics from one establish_key() call."""
    rounds: int
    photons_sent: int
    sifted_bits: int
    sample_size: int
    qber: float


class BB84QKDError(RuntimeError):
    """Raised when the estimated QBER exceeds the abort threshold, i.e.
    the channel is judged too noisy / too likely eavesdropped upon to
    safely extract a secret key (Sec. 2.2: 'if the error exceeds a
    predefined threshold, the protocol aborts')."""


class BB84QKD:
    """A single logical QKD link, simulated end-to-end with Qiskit.

    Each call to establish_key(length) runs one or more BB84 rounds
    (batched as large single-qubit-gate-only circuits) until enough
    sifted key material survives sifting + the public QBER sample, and
    returns the resulting shared secret key (identical bit string for
    both endpoints, matching KA = KB in the paper's Correctness
    definition, up to the negligible-eta case).
    """

    def __init__(self,
                 qber_threshold: float = 0.11,    # standard BB84 abort threshold
                 sample_ratio: float = 0.2,        # fraction of sifted bits sacrificed for QBER estimation
                 channel_error_rate: float = 0.0,  # simulated depolarizing noise / eavesdropping proxy
                 seed: Optional[int] = None,
                 link_name: str = "link"):
        self.qber_threshold = qber_threshold
        self.sample_ratio = sample_ratio
        self.channel_error_rate = channel_error_rate
        self._rng = random.Random(seed)
        self.backend = AerSimulator(method="matrix_product_state")
        self._noise_model = self._build_noise_model(channel_error_rate)
        self.link_name = link_name

    @staticmethod
    def _build_noise_model(error_rate: float) -> Optional[NoiseModel]:
        if error_rate <= 0.0:
            return None
        nm = NoiseModel()
        # A depolarizing channel applied on an explicit identity gate right
        # before Bob's measurement models a lossy/eavesdropped quantum
        # channel, giving Bob's outcome a chance to disagree with Alice's
        # even when their bases match.
        err = depolarizing_error(error_rate, 1)
        nm.add_all_qubit_quantum_error(err, ["id"])
        return nm

    # -- one BB84 round: state prep, transmission, measurement --------
    def _raw_round(self, n_photons: int, metrics: "Metrics" = None, phase: str = "kg"):
        alice_bits = [self._rng.randint(0, 1) for _ in range(n_photons)]
        alice_bases = [self._rng.randint(0, 1) for _ in range(n_photons)]   # 0 = Z, 1 = X
        bob_bases = [self._rng.randint(0, 1) for _ in range(n_photons)]

        # -- state preparation + circuit assembly (classical control logic) --
        classical_ctx = (metrics.time_block(phase, "classical") if metrics
                          else _null_context())
        with classical_ctx:
            qc = QuantumCircuit(n_photons, n_photons)
            for i in range(n_photons):
                if alice_bits[i] == 1:
                    qc.x(i)
                if alice_bases[i] == 1:
                    qc.h(i)
            if self._noise_model is not None:
                qc.id(range(n_photons))        # explicit noise-injection point (channel travel)
            qc.barrier()
            for i in range(n_photons):
                if bob_bases[i] == 1:
                    qc.h(i)
            qc.measure(range(n_photons), range(n_photons))

            run_kwargs = {"shots": 1}
            if self._noise_model is not None:
                run_kwargs["noise_model"] = self._noise_model

        # -- quantum transmission + measurement, simulated on Aer --
        quantum_ctx = (metrics.time_block(phase, "quantum") if metrics
                       else _null_context())
        with quantum_ctx:
            result = self.backend.run(qc, **run_kwargs).result()
            bitstring = next(iter(result.get_counts().keys()))
            # Qiskit's classical-register bitstring is ordered c[n-1] ... c[0].
            bob_bits = [int(b) for b in reversed(bitstring)]

        if metrics is not None:
            # n_photons qubits actually crossed the quantum channel this round.
            metrics.record_comm(phase, self.link_name, quantum_qubits=n_photons)

        return alice_bits, alice_bases, bob_bases, bob_bits

    def establish_key(self, length: int, max_rounds: int = 200,
                       metrics: "Metrics" = None, phase: str = "kg") -> tuple:
        """Run BB84 (batched over as many rounds as needed) until a
        `length`-bit shared secret key survives sifting and public QBER
        estimation. Returns (key_bits, stats).

        Raises BB84QKDError if the estimated QBER exceeds the abort
        threshold (Sec. 2.2 correctness/robustness requirement).

        If `metrics` is given, records: quantum computation time (circuit
        simulation), classical computation time (circuit assembly,
        sifting, QBER check), qubits sent over the quantum channel, and
        classical bits exchanged for basis reconciliation + QBER sample
        reveal, all tagged under `phase` and this link's name.
        """
        sifted_alice, sifted_bob = [], []
        photons_sent = 0
        rounds = 0

        # Reserve extra sifted bits for the public parameter-estimation
        # sample, since those bits are consumed and cannot be reused as
        # key material once revealed.
        target_before_sampling = max(
            length + 1, int(length / (1 - self.sample_ratio)) + 4
        )

        while len(sifted_alice) < target_before_sampling and rounds < max_rounds:
            remaining = target_before_sampling - len(sifted_alice)
            # ~50% of random bases match -> roughly double the photons needed.
            n_photons = max(32, int(remaining / 0.5 * 1.25))
            a_bits, a_bases, b_bases, b_bits = self._raw_round(n_photons, metrics, phase)
            photons_sent += n_photons
            with (metrics.time_block(phase, "classical") if metrics else _null_context()):
                for i in range(n_photons):
                    if a_bases[i] == b_bases[i]:
                        sifted_alice.append(a_bits[i])
                        sifted_bob.append(b_bits[i])
            rounds += 1

        if len(sifted_alice) < target_before_sampling:
            raise BB84QKDError(
                f"Could not sift enough key material for a {length}-bit key "
                f"within {max_rounds} rounds."
            )

        with (metrics.time_block(phase, "classical") if metrics else _null_context()):
            # --- Sifting: Alice and Bob publicly announce their bases
            # (1 bit/photon each way) over the classical channel. ---
            if metrics is not None:
                metrics.record_comm(phase, self.link_name,
                                     classical_bits=2 * photons_sent)

            # --- Parameter estimation (public sample -> QBER) ---
            n_sift = len(sifted_alice)
            sample_size = max(1, int(n_sift * self.sample_ratio))
            indices = list(range(n_sift))
            self._rng.shuffle(indices)
            sample_idx = set(indices[:sample_size])

            mismatches = sum(1 for i in sample_idx if sifted_alice[i] != sifted_bob[i])
            qber = mismatches / sample_size

            if metrics is not None:
                # Revealing the sampled bit values, both directions.
                metrics.record_comm(phase, self.link_name,
                                     classical_bits=2 * sample_size)

            if qber > self.qber_threshold:
                raise BB84QKDError(
                    f"Estimated QBER {qber:.3f} exceeds threshold "
                    f"{self.qber_threshold}; aborting key establishment "
                    f"(possible eavesdropping or excessive channel noise)."
                )

            # --- Key extraction: remaining (unsampled) sifted bits ---
            key_alice = [sifted_alice[i] for i in indices[sample_size:]][:length]

            stats = BB84Stats(
                rounds=rounds,
                photons_sent=photons_sent,
                sifted_bits=n_sift,
                sample_size=sample_size,
                qber=qber,
            )
        return key_alice[:length], stats


class QKDLink:
    """Thin wrapper exposing one BB84 session as the QKD primitive used
    by quPSI.kg to establish K_J, K_D, and K_JD. Each call to
    establish_key(length) runs an independent BB84 execution -- real
    state preparation, transmission-channel simulation, measurement,
    sifting, and QBER-based abort -- and returns a single shared key
    (KA = KB, per the paper's Correctness definition in Sec. 2.2).
    """

    def __init__(self, channel_error_rate: float = 0.0,
                 qber_threshold: float = 0.11, seed=None, link_name: str = "link"):
        self._bb84 = BB84QKD(qber_threshold=qber_threshold,
                              channel_error_rate=channel_error_rate,
                              seed=seed, link_name=link_name)

    def establish_key(self, length: int, metrics: "Metrics" = None,
                       phase: str = "kg") -> list:
        key, _stats = self._bb84.establish_key(length, metrics=metrics, phase=phase)
        return key


# =============================================================================
# PART B -- CLASSICAL LAYER: Boolean AND, Bloom filters, quPSI protocol
# =============================================================================

# --- Random oracle / hash utilities (Sec. 2, Sec. 4.2 assumption) ----------

def H(data: bytes) -> bytes:
    """Cryptographically secure hash function H: {0,1}* -> {0,1}^lambda.
    Used to compute the masked elements  x_bar_i = H(x_i || K_JD).
    """
    return hashlib.sha256(data).digest()


def make_hash_family(k: int, q: int, salt: bytes = b"quPSI") -> list:
    """Return k independent hash functions h_0,...,h_{k-1}, each mapping
    {0,1}* -> {0,1,...,q-1}, modeled as a random oracle (Sec. 2.3 / 4.2).
    Independence is obtained the standard random-oracle way: domain
    separation by prefixing each call with its index l before hashing.
    """
    def make_hl(l: int):
        def hl(x: bytes) -> int:
            digest = hashlib.sha256(salt + l.to_bytes(4, "big") + x).digest()
            return int.from_bytes(digest, "big") % q
        return hl

    return [make_hl(l) for l in range(k)]


def xor_bits(a: list, b: list) -> list:
    """Bitwise XOR of two equal-length bit arrays (list[int] of 0/1)."""
    assert len(a) == len(b), "XOR requires equal-length bit strings"
    return [x ^ y for x, y in zip(a, b)]


def _to_bytes(x) -> bytes:
    return x.encode("utf-8") if isinstance(x, str) else bytes(x)


def bits_to_bytes(bits: list) -> bytes:
    """Pack a 0/1 bit list into bytes (for use as hash input, e.g. K_JD)."""
    bit_str = "".join(str(b) for b in bits)
    pad = (-len(bit_str)) % 8
    bit_str += "0" * pad
    return int(bit_str, 2).to_bytes(len(bit_str) // 8, "big") if bit_str else b""


# --- Bloom filter (Sec. 2.3, Theorem 1) ------------------------------------

class BloomFilter:
    """Bloom filter BF_S parameterized by (q, n, k, H) as in Sec. 2.3.
    Backed by a plain list of 0/1 ints so it can be XORed with a
    one-time-pad key bit-for-bit, exactly as the protocol requires.
    """

    def __init__(self, q: int, hash_functions: list):
        self.q = q
        self.hash_functions = hash_functions
        self.bits = [0] * q

    @property
    def k(self) -> int:
        return len(self.hash_functions)

    def insert(self, element: bytes) -> None:
        """Insertion Phase (Sec. 2.3): set BF_S[h_l(x)] = 1 for all l."""
        for hl in self.hash_functions:
            self.bits[hl(element)] = 1

    def insert_all(self, elements: Iterable[bytes]) -> None:
        for e in elements:
            self.insert(e)

    def contains(self, element: bytes) -> bool:
        """Membership Test Phase (Sec. 2.3): z in S iff BF_S[h_l(z)] = 1
        for all l = 0,...,k-1 (possible false positive, never a false
        negative).
        """
        return all(self.bits[hl(element)] == 1 for hl in self.hash_functions)

    @staticmethod
    def optimal_length(max_set_size: int, k: int) -> int:
        """Theorem 1: q = n*k / ln(2), here n = max(|X|, |Y|), used to
        build a single shared array length for both parties (Sec. 3.1).
        """
        return max(1, math.ceil((max_set_size * k) / math.log(2)))

    def xor_with_key(self, key_bits: list) -> list:
        return xor_bits(self.bits, key_bits)

    @staticmethod
    def from_bits(bits: list, hash_functions: list) -> "BloomFilter":
        bf = BloomFilter(len(bits), hash_functions)
        bf.bits = list(bits)
        return bf

    @staticmethod
    def boolean_and(bf_x: "BloomFilter", bf_y: "BloomFilter") -> "BloomFilter":
        """f_AND (Sec. 2.1): entrywise Boolean AND, computed by TA on the
        recovered BF_X and BF_Y to produce BF_AND (Sec. 3.1, quPSI.int).
        """
        assert bf_x.q == bf_y.q
        and_bits = [1 if (x == 1 and y == 1) else 0
                    for x, y in zip(bf_x.bits, bf_y.bits)]
        return BloomFilter.from_bits(and_bits, bf_x.hash_functions)


# --- Parties and the Trusted Authority (Sec. 3, Sec. 4) --------------------

@dataclass
class TrustedAuthority:
    """T_A: semi-trusted third party. It only ever receives XOR-masked
    Bloom filters and only ever learns BF_AND; it never sees X, Y,
    BF_X, or BF_Y in the clear.
    """

    K_J: list = field(default=None)   # secret key shared with John
    K_D: list = field(default=None)   # secret key shared with David

    def compute_and_forward(self, masked_bfx: list, masked_bfy: list,
                             hash_functions: list,
                             metrics: "Metrics" = None) -> list:
        """quPSI.int, TA's steps:
          1. Retrieve BF_X = (BF_X xor K_J) xor K_J, BF_Y = (BF_Y xor K_D) xor K_D.
          2. Evaluate BF_AND = f_AND(BF_X, BF_Y) and return it (to be sent
             to both John and David).

        Records the TA's local computation time and the two outbound
        BF_AND deliveries (TA->John, TA->David), each q bits.
        """
        with (metrics.time_block("int", "classical") if metrics else _null_context()):
            bfx_bits = xor_bits(masked_bfx, self.K_J)
            bfy_bits = xor_bits(masked_bfy, self.K_D)
            bf_x = BloomFilter.from_bits(bfx_bits, hash_functions)
            bf_y = BloomFilter.from_bits(bfy_bits, hash_functions)
            bf_and = BloomFilter.boolean_and(bf_x, bf_y)

        if metrics is not None:
            q = len(bf_and.bits)
            metrics.record_comm("int", "TA-John", classical_bits=q)
            metrics.record_comm("int", "TA-David", classical_bits=q)

        return bf_and.bits


@dataclass
class Party:
    """A generic participant (John or David)."""

    name: str
    private_set: list                       # list of str/bytes elements
    K_own: list = field(default=None)       # key shared with TA (K_J or K_D)
    K_shared: list = field(default=None)    # K_JD, shared with the other party
    masked_set: list = field(default=None)  # X_bar or Y_bar (list[bytes])
    bloom_filter: BloomFilter = field(default=None)

    def mask_set(self, metrics: "Metrics" = None) -> None:
        """quPSI.int, step 1: x_bar_i = H(x_i || K_JD) for each element."""
        with (metrics.time_block("int", "classical") if metrics else _null_context()):
            kjd_bytes = bits_to_bytes(self.K_shared)
            self.masked_set = [H(_to_bytes(x) + kjd_bytes) for x in self.private_set]

    def build_bloom_filter(self, q: int, hash_functions: list,
                            metrics: "Metrics" = None) -> None:
        """quPSI.int, step 2: construct BF over the masked set."""
        with (metrics.time_block("int", "classical") if metrics else _null_context()):
            self.bloom_filter = BloomFilter(q, hash_functions)
            self.bloom_filter.insert_all(self.masked_set)

    def masked_bloom_filter_for_ta(self, metrics: "Metrics" = None,
                                    link_name: str = "Party-TA") -> list:
        """quPSI.int, step 3: BF_X xor K_J (or BF_Y xor K_D), sent to TA.
        Records the XOR-masking compute time and the q-bit transmission
        to the TA over the classical channel.
        """
        with (metrics.time_block("int", "classical") if metrics else _null_context()):
            masked = self.bloom_filter.xor_with_key(self.K_own)
        if metrics is not None:
            metrics.record_comm("int", link_name, classical_bits=len(masked))
        return masked

    def recover_intersection(self, bf_and_bits: list, hash_functions: list,
                              metrics: "Metrics" = None) -> set:
        """quPSI.comp: locally determine {x_i in X | BF_AND[h_l(x_bar_i)] = 1
        for all l}, then map back to the original elements.
        """
        with (metrics.time_block("comp", "classical") if metrics else _null_context()):
            bf_and = BloomFilter.from_bits(bf_and_bits, hash_functions)
            result = set()
            for original, masked in zip(self.private_set, self.masked_set):
                if bf_and.contains(masked):
                    result.add(original)
        return result


# --- The quPSI protocol driver ---------------------------------------------

@dataclass
class QuPSIResult:
    john_intersection: set
    david_intersection: set
    q: int
    k: int
    metrics: Optional["Metrics"] = None
    wall_clock_seconds: Optional[float] = None


def run_quPSI(X: list, Y: list, k: int = 20, verbose: bool = False,
              channel_error_rate: float = 0.0, qkd_seed=None,
              collect_metrics: bool = True,
              classical_bandwidth_bps: float = DEFAULT_CLASSICAL_BANDWIDTH_BPS,
              quantum_channel_rate_hz: float = DEFAULT_QUANTUM_CHANNEL_RATE_HZ) -> QuPSIResult:
    """Runs the full quPSI protocol (quPSI.kg -> quPSI.int -> quPSI.comp)
    between John (private set X) and David (private set Y), with an
    honest-but-curious Trusted Authority T_A, exactly as specified in
    Section 3.1. QKD keys (K_J, K_D, K_JD) are established with real
    BB84 sessions simulated via Qiskit (Part A above).

    Args:
        X, Y: the two parties' private input sets (lists of str/bytes).
        k:    number of independent hash functions (security/FP parameter).
              The paper recommends k >> max(n, m) for a negligible false
              positive probability max(n,m)/2^k (see Claim 2, Sec. 3.1).
        verbose: print intermediate protocol state for inspection.
        channel_error_rate: simulated noise/eavesdropping level on each
              BB84 quantum channel (0.0 = ideal channel). If the
              resulting QBER exceeds the abort threshold, key
              establishment raises BB84QKDError and the protocol halts.
        qkd_seed: optional seed for reproducible BB84 randomness.
        collect_metrics: if True (default), instrument the run and
              return a Metrics object on QuPSIResult.metrics (see
              print_performance_report). Set False to skip the small
              timing/bookkeeping overhead.
        classical_bandwidth_bps, quantum_channel_rate_hz: bandwidth model
              used only to turn recorded communication *size* into an
              *estimated* communication *time* (see Metrics).

    Returns:
        QuPSIResult with both parties' locally-recovered intersections
        (which are equal to X ∩ Y except with negligible probability),
        plus (when collect_metrics=True) a populated `metrics` field and
        the overall `wall_clock_seconds` for the whole run.
    """
    run_t0 = time.perf_counter()
    metrics = Metrics(classical_bandwidth_bps=classical_bandwidth_bps,
                       quantum_channel_rate_hz=quantum_channel_rate_hz) \
        if collect_metrics else None

    n, m = len(X), len(Y)
    q = BloomFilter.optimal_length(max(n, m), k)
    hash_functions = make_hash_family(k, q)

    john = Party(name="John", private_set=list(X))
    david = Party(name="David", private_set=list(Y))
    ta = TrustedAuthority()

    # --- quPSI.kg: Key Generation phase (Sec. 3.1) ---
    # Three independent BB84 (Qiskit-simulated) QKD sessions:
    # (John,TA), (David,TA), (John,David).
    link_jt = QKDLink(channel_error_rate=channel_error_rate, seed=qkd_seed, link_name="John-TA")
    link_dt = QKDLink(channel_error_rate=channel_error_rate, seed=qkd_seed, link_name="David-TA")
    link_jd = QKDLink(channel_error_rate=channel_error_rate, seed=qkd_seed, link_name="John-David")
    K_J = link_jt.establish_key(q, metrics=metrics, phase="kg")
    K_D = link_dt.establish_key(q, metrics=metrics, phase="kg")
    K_JD = link_jd.establish_key(q, metrics=metrics, phase="kg")

    john.K_own, john.K_shared = K_J, K_JD
    david.K_own, david.K_shared = K_D, K_JD
    ta.K_J, ta.K_D = K_J, K_D

    # --- quPSI.int: Initialization phase (Sec. 3.1) ---
    john.mask_set(metrics=metrics)
    david.mask_set(metrics=metrics)
    john.build_bloom_filter(q, hash_functions, metrics=metrics)
    david.build_bloom_filter(q, hash_functions, metrics=metrics)

    masked_bfx = john.masked_bloom_filter_for_ta(metrics=metrics, link_name="John-TA")
    masked_bfy = david.masked_bloom_filter_for_ta(metrics=metrics, link_name="David-TA")

    bf_and_bits = ta.compute_and_forward(masked_bfx, masked_bfy, hash_functions, metrics=metrics)

    # --- quPSI.comp: Computation phase (Sec. 3.1) ---
    john_result = john.recover_intersection(bf_and_bits, hash_functions, metrics=metrics)
    david_result = david.recover_intersection(bf_and_bits, hash_functions, metrics=metrics)

    if verbose:
        print(f"n={n}, m={m}, k={k}, q={q}")
        print("BF_X:", john.bloom_filter.bits)
        print("BF_Y:", david.bloom_filter.bits)
        print("BF_AND:", bf_and_bits)
        print("John recovers:", john_result)
        print("David recovers:", david_result)

    wall_clock = time.perf_counter() - run_t0

    return QuPSIResult(john_intersection=john_result,
                        david_intersection=david_result,
                        q=q, k=k, metrics=metrics,
                        wall_clock_seconds=wall_clock)


# =============================================================================
# Demo / self-test
# =============================================================================

def _demo():
    print("=" * 70)
    print("quPSI demo -- reproducing the paper's toy example (Sec. 3.2)")
    print("(K_J, K_D, K_JD are each established via a real BB84 session")
    print(" simulated with Qiskit: state prep, measurement, sifting,")
    print(" and QBER-based abort checking)")
    print("=" * 70)
    X = ["book", "pencil"]
    Y = ["pen", "book", "colour"]
    result = run_quPSI(X, Y, k=5, verbose=True, qkd_seed=1)

    expected = {"book"}
    assert result.john_intersection == expected, result.john_intersection
    assert result.david_intersection == expected, result.david_intersection
    print("\nOK: both parties independently recovered X ∩ Y =", result.john_intersection)
    print(f"\nEnd-to-end wall-clock time: {result.wall_clock_seconds:.4f} s")
    print_performance_report(result.metrics)

    print("\n" + "=" * 70)
    print("quPSI demo -- larger example (still BB84-backed QKD)")
    print("=" * 70)
    X2 = [f"suspect_{i}" for i in range(15)]
    Y2 = [f"suspect_{i}" for i in range(10, 25)]
    expected2 = set(X2) & set(Y2)
    result2 = run_quPSI(X2, Y2, k=16, qkd_seed=2)
    print(f"|X|={len(X2)}, |Y|={len(Y2)}, expected |X∩Y|={len(expected2)}")
    print("John's result matches expected:", result2.john_intersection == expected2)
    print("David's result matches expected:", result2.david_intersection == expected2)
    assert result2.john_intersection == expected2
    assert result2.david_intersection == expected2
    print("OK: exact match (k=16 hash functions makes false positives negligible).")
    print(f"\nEnd-to-end wall-clock time: {result2.wall_clock_seconds:.4f} s")
    print_performance_report(result2.metrics)

    print("\n" + "=" * 70)
    print("quPSI demo -- noisy/eavesdropped QKD channel triggers an abort")
    print("=" * 70)
    try:
        run_quPSI(["a", "b"], ["b", "c"], k=5, channel_error_rate=0.4, qkd_seed=3)
        print("(unexpectedly completed without aborting)")
    except BB84QKDError as e:
        print("Correctly aborted key establishment:", e)


if __name__ == "__main__":
    _demo()


quPSI demo -- reproducing the paper's toy example (Sec. 3.2)
(K_J, K_D, K_JD are each established via a real BB84 session
 simulated with Qiskit: state prep, measurement, sifting,
 and QBER-based abort checking)
n=2, m=3, k=5, q=22
BF_X: [0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
BF_Y: [1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1]
BF_AND: [0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
John recovers: {'book'}
David recovers: {'book'}

OK: both parties independently recovered X ∩ Y = {'book'}

End-to-end wall-clock time: 0.1617 s
------------------------------------------------------------------------------------------
Phase                                       Comp (s)      Qubits       Bytes      Comm (s)
------------------------------------------------------------------------------------------
quPSI.kg (BB84 key generation)                0.1591         231          64      0.000023
quPSI.int (masked BF exchange via TA)